In [1]:
!pip install nlpaug
import nlpaug.augmenter.word as naw
from datasets import load_dataset, concatenate_datasets, Dataset, DatasetDict
import random
import numpy as np

# --- (Paste your corrected get_augmented_train_set function here) ---
def get_augmented_train_set(train_ds, seed=42):
    """
    Takes an original train dataset and returns a new,
    augmented train dataset in a reproducible way.
    """
    random.seed(seed)
    np.random.seed(seed)

    def add_tracking_col(example):
        example['is_augmented'] = False
        return example
    train_ds = train_ds.map(add_tracking_col)

    aug = naw.RandomWordAug(action="delete")

    classes_to_augment = {
        "Ambivalent": 0.25,
        "Clear Reply": 0.25,
        "Clear Non-Reply": 0.25
    }

    new_rows_list = []
    print("Starting augmentation process...")

    for label, percentage in classes_to_augment.items():
        print(f"\n--- Augmenting class: {label} ---")

        class_subset = train_ds.filter(lambda x: x['clarity_label'] == label)
        num_original = len(class_subset)
        num_to_generate = int(num_original * percentage)

        if num_to_generate == 0:
            continue

        print(f"Original rows: {num_original}. Generating {num_to_generate} new rows.")
        indices_to_augment = random.choices(range(num_original), k=num_to_generate)
        texts_to_augment = [class_subset[i]['interview_answer'] for i in indices_to_augment]
        augmented_texts = aug.augment(texts_to_augment)

        for i, aug_text in enumerate(augmented_texts):
            original_row_index = indices_to_augment[i]
            original_row = class_subset[original_row_index]

            new_row = original_row.copy()
            new_row['interview_answer'] = aug_text
            new_row['is_augmented'] = True
            new_rows_list.append(new_row)

    print("\n--- Augmentation Complete ---")

    new_rows_ds = Dataset.from_list(new_rows_list, features=train_ds.features)
    augmented_train_ds = concatenate_datasets([train_ds, new_rows_ds])

    print(f"Final augmented train size: {len(augmented_train_ds)}")
    return augmented_train_ds

# -----------------------------------------------------------
# 2. Main script logic (MODIFIED)
# -----------------------------------------------------------
if __name__ == "__main__":
    print("Loading original dataset from Hugging Face...")
    original_dataset = load_dataset("ailsntua/QEvasion")

    print("Augmenting the 'train' split...")
    augmented_train_split = get_augmented_train_set(original_dataset['train'])

    # --- NEW CODE BLOCK ---
    print("Adding 'is_augmented' column to the 'test' split for schema consistency...")
    def add_tracking_col_test(example):
        example['is_augmented'] = False
        return example
    # We must add the column to the test split as well
    original_test_split = original_dataset['test'].map(add_tracking_col_test)
    # --- END NEW CODE BLOCK ---

    print("Creating new DatasetDict...")
    # Combine your new train split with the MODIFIED test split
    final_dataset = DatasetDict({
        'train': augmented_train_split,
        'test': original_test_split  # <-- Use the modified test split
    })

    # 3. Save the new dataset to disk
    output_dir = "./augmented_qevasion_dataset"
    print(f"Saving final dataset to {output_dir}...")
    final_dataset.save_to_disk(output_dir)

    print("Done. You can now run your training script.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 23.7 MB/s eta 0:00:00
Loading original dataset from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.90M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

...

Map:   0%|          | 0/308 [00:00<?, ? examples/s]

Creating new DatasetDict...
Saving final dataset to ./augmented_qevasion_dataset...


Saving the dataset (0/1 shards):   0%|          | 0/4310 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/308 [00:00<?, ? examples/s]

Done. You can now run your training script.


In [2]:
from datasets import load_dataset, load_from_disk
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, confusion_matrix # Added import

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the dataset
dataset = load_from_disk("./augmented_qevasion_dataset")
print("Loaded augmented dataset from disk.")

# Prepare labels
labels = dataset['train'].unique('clarity_label')
num_labels = len(labels)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

def add_labels(example):
    example['labels'] = label2id[example['clarity_label']]
    return example

dataset = dataset.map(add_labels)
dataset = dataset.remove_columns([
    col for col in dataset['train'].column_names if col not in ['question', 'interview_answer', 'labels']
])

print("Dataset ready:")
print(dataset)
print(f"Labels mapped: {label2id}")


# Calculate class weights for imbalanced data
def get_class_weights(dataset, num_labels):
    label_counts = Counter(dataset["train"]["labels"])
    total_samples = len(dataset["train"])

    # Calculate class weights (inverse frequency)
    class_weights = []
    for i in range(num_labels):
        count = label_counts.get(i, 1)  # avoid division by zero
        weight = total_samples / (num_labels * count)
        class_weights.append(weight)

    # Move the tensor to the active device (GPU)
    return torch.tensor(
        class_weights, dtype=torch.float32, device=device
    )


# Get class weights
class_weights = get_class_weights(dataset, num_labels)
print(f"Class weights: {class_weights}")
print(f"Class weights device: {class_weights.device}")  # Verify it's on CUDA

# Focal Loss implementation :cite[1]:cite[8]
class FocalLoss(nn.Module):
    """
    Multi-class Focal loss implementation
    Focal loss helps address class imbalance by focusing on hard examples

    Args:
        gamma (float): Focusing parameter, higher values put more focus on hard examples
        weight (Tensor): Class weights tensor for handling imbalanced data
        ignore_index (int): Index to ignore in loss calculation
    """
    def __init__(self, gamma=1.0, weight=None, ignore_index=-100):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.ignore_index = ignore_index

    def forward(self, input, target):
        # Calculate cross entropy loss
        ce_loss = F.cross_entropy(input, target, reduction='none', weight=self.weight, ignore_index=self.ignore_index)

        # Get probabilities
        pt = torch.exp(-ce_loss)

        # Compute focal loss
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        return focal_loss.mean()


# Updated Custom Trainer with corrected compute_loss signature
class CustomTrainer(Trainer):
    """
    Custom trainer that uses Focal Loss with class weights
    This subclass overrides the compute_loss method to use our custom loss function
    """

    def __init__(self, *args, class_weights=None, focal_gamma=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.focal_loss = FocalLoss(gamma=focal_gamma, weight=class_weights)

    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        # Extract labels and run model forward pass
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Compute focal loss with class weights
        loss = self.focal_loss(logits, labels)

        # Handle return_outputs as required by the Trainer
        return (loss, outputs) if return_outputs else loss

# Tokenization and model setup
model_checkpoint = "answerdotai/ModernBERT-large"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(
        examples['question'],
        examples['interview_answer'],
        truncation=True,
        padding="max_length",
        max_length=1680
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)

    # Use macro averaging for balanced metrics across classes
    precision_macro = precision_score(labels, predictions, average='macro', zero_division=0)
    recall_macro = recall_score(labels, predictions, average='macro', zero_division=0)
    f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)

    # Keep weighted for comparison
    precision_weighted = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall_weighted = recall_score(labels, predictions, average='weighted', zero_division=0)
    f1_weighted = f1_score(labels, predictions, average='weighted', zero_division=0)

    acc = accuracy_score(labels, predictions)

    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision_macro': precision_macro,
        'precision_weighted': precision_weighted,
        'recall_macro': recall_macro,
        'recall_weighted': recall_weighted
    }

# --- MODIFICATION 1: Updated TrainingArguments ---
# We now evaluate, save, and load the best model based on 'f1_macro'
training_args = TrainingArguments(
    output_dir="ModernBERT_QEvasion_model",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    num_train_epochs=7,
    weight_decay=0.01,
    eval_strategy="epoch",          # <--- MODIFIED (was "no")
    save_strategy="epoch",
    load_best_model_at_end=True,    # <--- MODIFIED (was False)
    metric_for_best_model="f1_macro", # <--- NEW
    greater_is_better=True,         # <--- NEW
    push_to_hub=False,
    logging_steps=100,
    report_to="none",
    fp16=True,  # Enable mixed precision (reduces memory usage)
    gradient_checkpointing=True,
)

# --- MODIFICATION 2: Updated CustomTrainer instantiation ---
# We pass the test set to eval_dataset
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],  # <--- NEW
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights,  # Pass the calculated class weights
    focal_gamma=1.0,  # You can adjust this parameter
)

print(f"Using class weights: {class_weights}")
print("Starting training with Focal Loss (evaluating on test set after each epoch)...")
trainer.train()

print("Training completed!")

# --- MODIFICATION 3: Updated Final Evaluation ---
# This will now evaluate the *best* model saved during training
# (because of load_best_model_at_end=True)
test_results = trainer.evaluate() # <--- MODIFIED (no arg needed)
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (from best epoch: {trainer.state.best_model_checkpoint})") # <--- MODIFIED
print("="*60)
for key, value in test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# Optional: Get detailed predictions
# This will also use the best model
print("\nDetailed predictions analysis (from best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                          target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))


Using device: cuda
Loaded augmented dataset from disk.


Map:   0%|          | 0/4310 [00:00<?, ? examples/s]

Map:   0%|          | 0/308 [00:00<?, ? examples/s]

Dataset ready:
DatasetDict({
    train: Dataset({
        features: ['interview_answer', 'question', 'labels'],
        num_rows: 4310
    })
    test: Dataset({
        features: ['interview_answer', 'question', 'labels'],
        num_rows: 308
    })
})
Labels mapped: {'Clear Reply': 0, 'Ambivalent': 1, 'Clear Non-Reply': 2}
Class weights: tensor([1.0925, 0.5634, 3.2285], device='cuda:0')
Class weights device: cuda:0


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-514792795.py:99: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `CustomTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Using class weights: tensor([1.0925, 0.5634, 3.2285], device='cuda:0')
Starting training with Focal Loss (evaluating on test set after each epoch)...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted
1,0.559100,0.481398,0.603896,0.552887,0.617280,0.530678,0.647607,0.600743,0.603896
2,0.329300,0.435150,0.590909,0.526890,0.603699,0.502829,0.638475,0.617287,0.590909
3,0.241700,0.679094,0.532468,0.535310,0.546888,0.556408,0.637067,0.575813,0.532468
4,0.136900,0.825928,0.698052,0.605391,0.687219,0.612626,0.685045,0.608781,0.698052
5,0.076500,1.294374,0.665584,0.610295,0.666488,0.602874,0.667616,0.618481,0.665584
6,0.056500,1.581303,0.691558,0.618386,0.687571,0.619767,0.684908,0.618551,0.691558
7,0.010100,1.657252,0.704545,0.620475,0.700362,0.636728,0.697748,0.607078,0.704545


Training completed!



FINAL TEST RESULTS (from best epoch: ModernBERT_QEvasion_model/checkpoint-3773)
eval_loss: 1.6573
eval_accuracy: 0.7045
eval_f1_macro: 0.6205
eval_f1_weighted: 0.7004
eval_precision_macro: 0.6367
eval_precision_weighted: 0.6977
eval_recall_macro: 0.6071
eval_recall_weighted: 0.7045

Detailed predictions analysis (from best model):

Classification Report:
                 precision    recall  f1-score   support

    Clear Reply       0.54      0.49      0.52        79
     Ambivalent       0.77      0.81      0.79       206
Clear Non-Reply       0.60      0.52      0.56        23

       accuracy                           0.70       308
      macro avg       0.64      0.61      0.62       308
   weighted avg       0.70      0.70      0.70       308


Confusion Matrix:
[[ 39  39   1]
 [ 33 166   7]
 [  0  11  12]]
